# 🌿 03 — SAR + Optical Fusion Change Detection
### Amazon Deforestation — Multi-Sensor Change Detection

Runs DOFA on both optical and SAR channels separately, then fuses the
predictions using confidence-weighted voting for cloud-robust results.

**Fusion strategy:**
1. **Optical branch** — DOFA on RGB with Sentinel-2 wavelengths [0.665, 0.549, 0.481]
2. **SAR branch** — DOFA on VV+VH pseudo-RGB with SAR-equivalent wavelengths
3. **Confidence fusion** — weighted average of softmax probabilities
4. **Change detection** — compare fused 2019 vs 2024 predictions
5. **SAR-only change** — VH/VV ratio change as independent validation

**Key advantage over optical-only:**
- SAR works through clouds → detection works year-round
- SAR backscatter change (forest → bare soil) is highly discriminative
- Fusion reduces false positives from cloud shadows in optical


## ⚙️ Configuration

In [1]:
import sys
import numpy as np
import torch
import rasterio
import pandas as pd
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, r"C:\geo-deep-learning")

CHECKPOINT_PATH = Path(r"C:\geo-deep-learning\logs\gdl_experiment\version_11\checkpoints\model-epoch=00-val_loss=0.141.ckpt")
PREPARED_DIR    = Path("data/prepared_sar_optical")
OUTPUT_DIR      = Path("outputs/fusion_change")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE  = 64
STRIDE      = 48
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]

# Wavelengths for each sensor branch
OPT_WAVELENGTHS = torch.tensor([0.665, 0.549, 0.481])   # Sentinel-2 RGB

# SAR C-band: 5.6cm = 0.056m — DOFA accepts any wavelength
# We use the three closest visible wavelengths as proxies
# This is the standard approach for applying optical-pretrained models to SAR
SAR_WAVELENGTHS = torch.tensor([0.056, 0.056, 0.056])   # C-band proxy

CLASS_NAMES = [
    "Annual Crop", "Forest", "Herbaceous Veg", "Highway",
    "Industrial", "Pasture", "Permanent Crop", "Residential",
    "River", "Sea / Lake",
]
NUM_CLASSES = len(CLASS_NAMES)

PIXEL_SIZE_M       = 10.0
HECTARES_PER_PIXEL = (PIXEL_SIZE_M ** 2) / 10_000

print("Configuration set")
print(f"  Checkpoint : {CHECKPOINT_PATH.exists()}")
print(f"  Prepared   : {PREPARED_DIR.exists()}")


Configuration set
  Checkpoint : True
  Prepared   : True


## 📦 Step 1 — Load DOFA Model

In [2]:
import segmentation_models_pytorch as smp
from geo_deep_learning.tasks_with_models.segmentation_dofa import SegmentationDOFA

ckpt  = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
hp    = ckpt["hyper_parameters"]
model = SegmentationDOFA(
    encoder=hp["encoder"], pretrained=False,
    image_size=tuple(hp["image_size"]), num_classes=hp["num_classes"],
    max_samples=hp.get("max_samples", 2),
    loss=smp.losses.DiceLoss(mode="multiclass", from_logits=True),
    class_labels=hp.get("class_labels"), class_colors=hp.get("class_colors"),
    freeze_layers=hp.get("freeze_layers"),
)
model.configure_model()
state = {k.removeprefix("model."): v for k, v in ckpt["state_dict"].items()}
model.model.load_state_dict(state, strict=True)
model.eval()
print(f"Model loaded — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")


Model loaded — 140.3M params


## 🔍 Step 2 — Sliding Window Inference

In [3]:
tf = T.Compose([T.Resize((PATCH_SIZE, PATCH_SIZE)), T.Normalize(mean=MEAN, std=STD)])

def segment_scene(scene_array, wavelengths, label=""):
    """
    Run DOFA sliding-window inference on a (3, H, W) scene.
    Returns confidence map (H, W, C) and class map (H, W).
    """
    _, H, W   = scene_array.shape
    conf_map  = np.zeros((H, W, NUM_CLASSES), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)
    wv        = wavelengths.unsqueeze(0)
    n         = 0

    with torch.no_grad():
        for y in range(0, H - PATCH_SIZE + 1, STRIDE):
            for x in range(0, W - PATCH_SIZE + 1, STRIDE):
                chip    = scene_array[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                pil_img = Image.fromarray(
                    (chip.transpose(1,2,0)*255).clip(0,255).astype(np.uint8))
                tensor  = tf(T.ToTensor()(pil_img)).unsqueeze(0)
                out     = model(tensor, wv)
                probs   = out.out.softmax(dim=1).squeeze(0).permute(1,2,0).numpy()
                conf_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE]  += probs
                count_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += 1.0
                n += 1

    conf_map /= np.maximum(count_map, 1.0)[:, :, np.newaxis]
    cls_map   = conf_map.argmax(axis=2).astype(np.int32)
    max_conf  = conf_map.max(axis=2).mean()
    print(f"   {label}: {n} patches  mean_confidence={max_conf:.3f}")
    return conf_map, cls_map

def load_scene(year, sensor):
    with rasterio.open(PREPARED_DIR / f"{sensor}_{year}.tif") as src:
        return src.read(), src.profile

print("Loading scenes...")
opt_2019, profile = load_scene("2019", "optical")
opt_2024, _       = load_scene("2024", "optical")
sar_2019, _       = load_scene("2019", "sar")
sar_2024, _       = load_scene("2024", "sar")

# Align all to minimum common shape
H = min(opt_2019.shape[1], opt_2024.shape[1], sar_2019.shape[1], sar_2024.shape[1])
W = min(opt_2019.shape[2], opt_2024.shape[2], sar_2019.shape[2], sar_2024.shape[2])
opt_2019 = opt_2019[:3, :H, :W]
opt_2024 = opt_2024[:3, :H, :W]
sar_rgb_2019 = np.stack([sar_2019[0], sar_2019[1], sar_2019[2]], axis=0)[:, :H, :W]
sar_rgb_2024 = np.stack([sar_2024[0], sar_2024[1], sar_2024[2]], axis=0)[:, :H, :W]

print(f"Scenes loaded and aligned — shape: ({H}, {W})")
print()
print("Running optical inference...")
opt_conf_2019, opt_cls_2019 = segment_scene(opt_2019, OPT_WAVELENGTHS, "Optical 2019")
opt_conf_2024, opt_cls_2024 = segment_scene(opt_2024, OPT_WAVELENGTHS, "Optical 2024")

print()
print("Running SAR inference...")
sar_conf_2019, sar_cls_2019 = segment_scene(sar_rgb_2019, SAR_WAVELENGTHS, "SAR 2019")
sar_conf_2024, sar_cls_2024 = segment_scene(sar_rgb_2024, SAR_WAVELENGTHS, "SAR 2024")


Loading scenes...
Scenes loaded and aligned — shape: (512, 512)

Running optical inference...
   Optical 2019: 100 patches  mean_confidence=0.920
   Optical 2024: 100 patches  mean_confidence=0.572

Running SAR inference...
   SAR 2019: 100 patches  mean_confidence=0.922
   SAR 2024: 100 patches  mean_confidence=0.530


## 🔀 Step 3 — Confidence-Weighted Fusion

In [4]:
# Fuse optical and SAR predictions using confidence weighting
# Optical gets higher weight (0.6) as it was used in training
# SAR gets lower weight (0.4) as it is out-of-distribution for DOFA
OPT_WEIGHT = 0.65
SAR_WEIGHT = 0.35

print(f"Fusing predictions (optical={OPT_WEIGHT}, SAR={SAR_WEIGHT})...")

fused_conf_2019 = OPT_WEIGHT * opt_conf_2019 + SAR_WEIGHT * sar_conf_2019
fused_conf_2024 = OPT_WEIGHT * opt_conf_2024 + SAR_WEIGHT * sar_conf_2024
fused_cls_2019  = fused_conf_2019.argmax(axis=2).astype(np.int32)
fused_cls_2024  = fused_conf_2024.argmax(axis=2).astype(np.int32)

# Binary change map
binary_change = (fused_cls_2019 != fused_cls_2024).astype(np.uint8)

# SAR-only change — independent validation using VH/VV ratio
# Forest has higher VH/VV than bare soil
sar_ratio_2019 = sar_2019[2, :H, :W]   # VH/VV ratio band
sar_ratio_2024 = sar_2024[2, :H, :W]
sar_ratio_diff = sar_ratio_2019 - sar_ratio_2024   # positive = vegetation loss
sar_change_mask = (sar_ratio_diff > 0.15).astype(np.uint8)  # threshold

# Combined: change detected by BOTH fusion AND SAR ratio
confirmed_change = ((binary_change == 1) & (sar_change_mask == 1)).astype(np.uint8)

total     = H * W
n_changed = int(binary_change.sum())
n_sar     = int(sar_change_mask.sum())
n_conf    = int(confirmed_change.sum())
ha_fused  = n_changed * HECTARES_PER_PIXEL
ha_conf   = n_conf   * HECTARES_PER_PIXEL

print()
print("Change Detection Results:")
print("-" * 45)
print(f"  Fusion-detected change : {n_changed:,} px  ({n_changed/total*100:.1f}%)  {ha_fused:,.0f} ha")
print(f"  SAR-only change        : {n_sar:,} px  ({n_sar/total*100:.1f}%)")
print(f"  Confirmed (both)       : {n_conf:,} px  ({n_conf/total*100:.1f}%)  {ha_conf:,.0f} ha")


Fusing predictions (optical=0.65, SAR=0.35)...

Change Detection Results:
---------------------------------------------
  Fusion-detected change : 210,074 px  (80.1%)  2,101 ha
  SAR-only change        : 70,488 px  (26.9%)
  Confirmed (both)       : 55,616 px  (21.2%)  556 ha


## 💾 Step 4 — Save Outputs

In [5]:
def save_raster(arr, path, dtype, profile):
    if arr.ndim == 2: arr = arr[np.newaxis, :]
    p = profile.copy()
    p.update(count=arr.shape[0], dtype=dtype, driver="GTiff", compress="lzw",
             height=arr.shape[1], width=arr.shape[2])
    with rasterio.open(path, "w", **p) as dst:
        dst.write(arr)
    print(f"   Saved: {path.name}")

save_raster(fused_cls_2019,  OUTPUT_DIR/"fused_seg_2019.tif",  "int32",  profile)
save_raster(fused_cls_2024,  OUTPUT_DIR/"fused_seg_2024.tif",  "int32",  profile)
save_raster(opt_cls_2019,    OUTPUT_DIR/"optical_seg_2019.tif","int32",  profile)
save_raster(opt_cls_2024,    OUTPUT_DIR/"optical_seg_2024.tif","int32",  profile)
save_raster(sar_cls_2019,    OUTPUT_DIR/"sar_seg_2019.tif",    "int32",  profile)
save_raster(sar_cls_2024,    OUTPUT_DIR/"sar_seg_2024.tif",    "int32",  profile)
save_raster(binary_change,   OUTPUT_DIR/"binary_change.tif",   "uint8",  profile)
save_raster(confirmed_change,OUTPUT_DIR/"confirmed_change.tif","uint8",  profile)
save_raster(sar_ratio_diff,  OUTPUT_DIR/"sar_ratio_diff.tif",  "float32",profile)

# Transition table
changed = np.where(binary_change == 1)
from_cls, to_cls = fused_cls_2019[changed], fused_cls_2024[changed]
records = []
for fc in range(NUM_CLASSES):
    for tc in range(NUM_CLASSES):
        if fc == tc: continue
        n = int(((from_cls==fc)&(to_cls==tc)).sum())
        if n > 0:
            records.append({
                "from_class": CLASS_NAMES[fc], "to_class": CLASS_NAMES[tc],
                "pixels": n, "hectares": round(n * HECTARES_PER_PIXEL, 2),
            })
df = pd.DataFrame(records).sort_values("hectares", ascending=False)
df.to_csv(OUTPUT_DIR/"fusion_transitions.csv", index=False)
print(f"   Saved: fusion_transitions.csv  ({len(df)} transitions)")
print()
print("All outputs saved. Proceed to: 04_visualise_fusion.ipynb")


   Saved: fused_seg_2019.tif
   Saved: fused_seg_2024.tif
   Saved: optical_seg_2019.tif
   Saved: optical_seg_2024.tif
   Saved: sar_seg_2019.tif
   Saved: sar_seg_2024.tif
   Saved: binary_change.tif
   Saved: confirmed_change.tif
   Saved: sar_ratio_diff.tif
   Saved: fusion_transitions.csv  (6 transitions)

All outputs saved. Proceed to: 04_visualise_fusion.ipynb
